Пишем код для учета атмосферной рефракции в приближении плоско-параллельной атмосферы

In [16]:
from math import *
import numpy as np
from datetime import datetime
from astropy.time import Time

from astropy.coordinates import EarthLocation, AltAz, SkyCoord
import astropy.units as u
from astropy.time import Time

def tangFromRADE(ra, dec, RA, DEC):
    ksi = cos(dec)*sin(ra-RA)/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    eta = (sin(dec)*cos(DEC)-cos(dec)*sin(DEC)*cos(ra-RA))/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    return ksi,eta

def RADecFromTang(ksi, eta, RA, Dec):
    x,y,z = np.dot(np.array([[-sin(RA),-cos(RA) * sin(Dec),cos(RA) * cos(Dec)],
                             [cos(RA),-sin(RA) * sin(Dec),sin(RA) * cos(Dec)],
                             [0,cos(Dec),sin(Dec)]]),
                   np.array([ksi,eta,1]))/sqrt(1+ksi*ksi+eta*eta)
    ra = atan2(y,x)
    dec = atan2(z,sqrt(x*x+y*y))
    if(ra<0):
        ra+=2*pi
    return ra,dec
# это функция для вычисления звездного времени
def getSiderial(JD, lon):
    MJD = JD - 2400000.5
    H = (MJD - trunc(MJD))*24.0
    D = JD - 2451545.0
    D0 = trunc(MJD) - 51544.5
    T = D/36525
    LST = 6.697374558 + 0.06570982441908*D0 + 1.00273790935*H + 0.000026*T*T+degrees(lon)/15
    return pi*(LST - 24.0*trunc(LST/24.0))/12.0

def RADECtoAzEl(ra,dec,s,lat):
    Phi = lat-pi/2.0
    e = np.array([cos(s-ra)*cos(dec),sin(s-ra)*cos(dec),sin(dec)])
    h = np.array([cos(Phi)*e[0]+sin(Phi)*e[2],e[1],-sin(Phi)*e[0]+cos(Phi)*e[2]])
    cosd = sqrt(h[0] * h[0] + h[1] * h[1])
    return atan2(h[1] / cosd, h[0] / cosd), atan2(h[2], cosd)

def AzEltoRADEC(A, h, s, lat):
    sin_dec = np.sin(h)*np.sin(lat) + np.cos(h)*np.cos(lat) * np.cos(A)
    dec = np.arcsin(sin_dec)
    ra = s - np.arctan2(-np.sin(A)*np.cos(h), \
                         np.sin(h)*np.cos(lat) - np.cos(h)*np.sin(lat)*np.cos(A))
    return ra, dec

In [12]:
def refraction(ra,dec, s, lat, JD, t, P, wl):
    az, h = RADECtoAzEl(ra, dec, s, lat)
    zeta = np.pi/2.0 - h # from zenith
    
    k0 = 2.871e-4 * (1 + 0.00567 / wl**2)
    k = k0 * (P/760) * (273 / (273+t))
    rho = k * np.tan(zeta)
    
    
    
    return ra_ref.to(u.rad).value, dec_ref.to(u.rad).value


In [17]:
lon = radians(30.327498)
lat = radians(59.771831)
P = 750 # mmhg
t = 7 # celsius
wl = 0.55 # mcm
ra,dec = radians(60.0),radians(-20.0)
JD = float(Time('2026-05-27 23:00:00').copy(format='jd').value)
print(JD)

2461188.4583333335


In [20]:
A, h = RADECtoAzEl(np.radians(50.0), np.radians(-20.0), np.pi/2.0, np.radians(35.0))
A, h

ra, dec = AzEltoRADEC(A, h, np.pi/2.0, np.radians(35.0))
np.degrees(ra), np.degrees(dec)

(187.0959697007272, 52.50552549792499)

In [132]:
# просто потренируйтесь вычислять координаты, исправленные за влияние атмосферной рефракции
ra,dec = 30,30
s = getSiderial(JD,lon)
ra_ref, dec_ref = refraction(ra,dec,s,lat,JD,t,P,wl) * u.rad
ra_ref.to(u.deg).value, dec_ref.to(u.deg).value

-4.37561037329883 deg
-51.24640152260092 deg
-0.7765531318923893 arcmin


ValueError: Latitude angle(s) must be within -90 deg <= angle <= 90 deg, got 141.2334589704027 deg

Допустим звезда имеет координаты ra,dec = s,30 (s - звездное время) в начальный момент наблюдений. Околоземный астероид проходит вблизи звезды и его координаты меняются линейно со временем. В момент s тангенциальные координаты астероида составляют $\xi,\eta = 5,5\, arcsec$ без учета атмосферной рефракции. Компоненты скорости астероида $\dot \xi, \dot \eta = 24,17 \,arcsec/hour$. Звезда имеет максимум излучения в спектре на длине волны $\lambda_s = 500\, нм$, астероид - $\lambda_a = 650\, нм$. Условия наблюдений ($P = 556\, мм.рт.ст., t = 3^\circ C$). Построить траектории астероида относительно звезды в тангенциальных координатах с учетом рефракции и без ее учета на протяжении трех часов после начала наблюдений. Оценить, насколько значим эффект атмосферной рефракции для точной астрометрии околоземных астероидов? 